# 23 · Tool calls on the wire, three ways
Level L2 · Part 3 · professional · ~30 min · builds on 10, 22

## Goal
Write one `search_notes` round trip as an Anthropic (`tool_use` / `tool_result`), an OpenAI
(`function_call` / `function_call_output`) and a Gemini Interactions (`function_call` / `function_result`)
request, and see what happens when a result is missing, has the wrong id or sits in the wrong place:
a documented 400 on Anthropic, a real 400 or silent misbehaviour on Gemini `generate_content`, and how to
catch all of them before sending.

## The idea
Lesson 22 did one tool round trip on Gemini: the model asks for a tool by name with an id, your code runs
it, and you send the result back with the same id. Every vendor follows that pattern but packages it
differently: where the call sits, what the id field is called, whether arguments are an object or a JSON
string, and where the result must go. Tool calls are where adapters break, because the rules about **pairing** (every call gets a result with its id) and **placement**
(the result comes right after the call) are strict on some APIs and silently loose on others.

Analogy: a claim ticket at a coat check. Each venue prints a different ticket, but you only get a
coat back if your ticket number matches one they gave out.

## Picture
```mermaid
flowchart LR
    H["Scout history<br/>user, tool_calls, tool"] --> A["Anthropic<br/>tool_use / tool_result<br/>tool_use_id"]
    H --> O["OpenAI Responses<br/>function_call / function_call_output<br/>call_id"]
    H --> G["Gemini Interactions<br/>function_call / function_result<br/>call_id"]
```

## Step by step
### 1. Start the recorder
The Gemini calls in this notebook are recorded once to `../data/llm_cassettes/23_tool-call-wire-formats.yaml`
and replayed after that, with no key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("23_tool-call-wire-formats")

LLM mode: replay (23_tool-call-wire-formats.yaml)


### 2. Setup from lessons 10 and 22
The notes, `search_notes` and its schema, the Gemini client and the tool config from lesson 22. Nothing new.

In [2]:
import json, os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    return [note for note in NOTES if query.lower() in note["text"].lower()]

TOOLS = {"search_notes": search_notes}
print(len(NOTES), "notes | tools:", list(TOOLS))

6 notes | tools: ['search_notes']


The tool schema, the Gemini client on Vertex and lesson 22's tool config with automatic calling off.

In [3]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}. Use it before answering any question "
                   "about what the notes say.",
    "parameters": {"type": "object", "required": ["query"],
                   "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}}},
}
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
search_decl = types.FunctionDeclaration(name="search_notes", description=SEARCH_NOTES_SCHEMA["description"],
                                        parameters_json_schema=SEARCH_NOTES_SCHEMA["parameters"])
TOOL_CFG = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[search_decl])],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True), max_output_tokens=300)
print("model:", MODEL, "| declared:", search_decl.name)

model: gemini-3.5-flash-lite | declared: search_notes


And lesson 10's checking trick: each vendor SDK gets an HTTP client whose transport answers every request
with a local 400, so the SDK builds the real request body and nothing leaves the machine.

In [4]:
import httpx2, anthropic, openai
# Model names from RESEARCH.md §1. They are never called: the local transport refuses every request.
VENDOR_MODELS = {"anthropic": "claude-sonnet-5-5", "openai": "gpt-6-luna", "gemini": "gemini-3.8-flash"}

def local_http() -> httpx2.Client:
    refuse = lambda request: httpx2.Response(400, json={"error": {"type": "invalid_request_error",
                                                                   "message": "not sent"}})
    return httpx2.Client(transport=httpx2.MockTransport(refuse))

def wire(send) -> dict:
    try:
        send()
    except Exception as err:                       # each SDK raises its own 400 error class
        req = err.response.request
        print(req.method, req.url)
        return json.loads(req.content)
    raise RuntimeError("expected the local 400")

### 3. One real tool call, in Scout's own format
We ask Gemini (the vendor with a key here) the lesson 22 question, run the tool it asks for, and store the
round trip in the vendor-neutral shape of lesson 22's fake model: a `user` turn, an `assistant` turn with
`tool_calls`, and a `tool` turn with `call_id`.

In [5]:
QUESTION = "Which of my notes mention Mars?"
contents = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]
resp1 = client.models.generate_content(model=MODEL, contents=contents, config=TOOL_CFG)
fc = resp1.function_calls[0]
real_call = {"id": fc.id, "name": fc.name, "args": dict(fc.args)}
print("Gemini asked for:", real_call)

Gemini asked for: {'id': 'call_465692', 'name': 'search_notes', 'args': {'query': 'Mars'}}


Now run the tool and build the neutral history. (If the model cell was skipped, a labelled placeholder
call stands in so the rest still runs.)

In [6]:
QUESTION = "Which of my notes mention Mars?"
if "real_call" not in globals():
    real_call = {"id": "call_placeholder", "name": "search_notes", "args": {"query": "Mars"}}   # placeholder
hits = TOOLS[real_call["name"]](**real_call["args"])
history = [
    {"role": "user", "text": QUESTION},
    {"role": "assistant", "tool_calls": [real_call]},
    {"role": "tool", "call_id": real_call["id"], "name": real_call["name"], "content": json.dumps(hits)},
]
for m in history:
    print(m["role"], "|", {k: v for k, v in m.items() if k != "role"} if m["role"] != "tool" else m["call_id"])

user | {'text': 'Which of my notes mention Mars?'}
assistant | {'tool_calls': [{'id': 'call_465692', 'name': 'search_notes', 'args': {'query': 'Mars'}}]}
tool | call_465692


### 4. A pairing check before any vendor sees it
The rule all three vendors share: every tool call is answered by exactly one result with its id, and the
results come right after the turn that made the calls. We check it once, on the neutral history:
the run of `tool` messages after a call turn must hold exactly the asked ids, no more, no fewer, no repeats.

In [7]:
def check_tool_pairs(messages: list[dict]) -> None:
    for i, m in enumerate(messages):
        prev = messages[i - 1] if i else {}
        if m["role"] == "tool" and not (prev.get("tool_calls") or prev.get("role") == "tool"):
            raise ValueError(f"message {i}: tool result {m['call_id']!r} does not follow a tool-call turn")
        if m.get("tool_calls"):
            asked = [c["id"] for c in m["tool_calls"]]
            answered = []                          # the whole run of tool messages after the call turn
            for r in messages[i + 1:]:
                if r["role"] != "tool":
                    break
                answered.append(r["call_id"])
            if sorted(answered) != sorted(asked):  # catches missing, extra and duplicate ids
                raise ValueError(f"message {i}: calls {asked} need results right after, got {answered}")

check_tool_pairs(history)
print("pairs OK:", real_call["id"], "->", history[2]["call_id"])

pairs OK: call_465692 -> call_465692


### 5. Anthropic: `tool_use` and `tool_result` blocks
- The tool schema's arguments go in **`input_schema`**.
- The model's call is a **`tool_use` content block** in an `assistant` message: `id`, `name`, `input` (an object).
- The result is a **`tool_result` block in a `user` message** (there is no `tool` role), linked by
  **`tool_use_id`**. Its `content` is a string or a list of blocks.

In [8]:
def to_anthropic_tools(messages: list[dict]) -> dict:
    check_tool_pairs(messages)
    out = []
    for m in messages:
        if m["role"] == "user":
            out.append({"role": "user", "content": [{"type": "text", "text": m["text"]}]})
        elif m["role"] == "assistant":
            out.append({"role": "assistant", "content": [{"type": "tool_use", "id": c["id"], "name": c["name"],
                                                          "input": c["args"]} for c in m["tool_calls"]]})
        else:
            block = {"type": "tool_result", "tool_use_id": m["call_id"], "content": m["content"]}
            if out[-1]["role"] == "user":       # several results share ONE user message
                out[-1]["content"].append(block)
            else:
                out.append({"role": "user", "content": [block]})
    tool = {"name": SEARCH_NOTES_SCHEMA["name"], "description": SEARCH_NOTES_SCHEMA["description"],
            "input_schema": SEARCH_NOTES_SCHEMA["parameters"]}
    return {"model": VENDOR_MODELS["anthropic"], "max_tokens": 300, "tools": [tool], "messages": out}

anthropic_req = to_anthropic_tools(history)
for m in anthropic_req["messages"]:
    print(m["role"], json.dumps(m["content"][0])[:95])

user {"type": "text", "text": "Which of my notes mention Mars?"}
assistant {"type": "tool_use", "id": "call_465692", "name": "search_notes", "input": {"query": "Mars"}}
user {"type": "tool_result", "tool_use_id": "call_465692", "content": "[{\"id\": 1, \"text\": \"Mars


### 6. OpenAI Responses: `function_call` and `function_call_output` items
- A tool is `{"type": "function", "name", "description", "parameters"}` (flat, no nesting).
- The call is its own **`function_call` item** in `input`, not inside a message. The id field is
  **`call_id`**, and **`arguments` is a JSON string**, not an object.
- The result is a **`function_call_output` item** with the same `call_id` and an `output` string.

In [9]:
def to_responses_tools(messages: list[dict]) -> dict:
    check_tool_pairs(messages)
    items = []
    for m in messages:
        if m["role"] == "user":
            items.append({"role": "user", "content": [{"type": "input_text", "text": m["text"]}]})
        elif m["role"] == "assistant":
            items += [{"type": "function_call", "call_id": c["id"], "name": c["name"],
                       "arguments": json.dumps(c["args"])} for c in m["tool_calls"]]
        else:
            items.append({"type": "function_call_output", "call_id": m["call_id"], "output": m["content"]})
    tool = {"type": "function", **SEARCH_NOTES_SCHEMA}
    return {"model": VENDOR_MODELS["openai"], "tools": [tool], "input": items}

responses_req = to_responses_tools(history)
for item in responses_req["input"]:
    print(item.get("type", item.get("role")), json.dumps(item)[:95])

user {"role": "user", "content": [{"type": "input_text", "text": "Which of my notes mention Mars?"}]
function_call {"type": "function_call", "call_id": "call_465692", "name": "search_notes", "arguments": "{\"qu
function_call_output {"type": "function_call_output", "call_id": "call_465692", "output": "[{\"id\": 1, \"text\": \"


### 7. Gemini Interactions: `function_call` and `function_result` steps
- A tool is `{"type": "function", "name", "description", "parameters"}`, like OpenAI.
- The call is a **`function_call` step** with `id`, `name` and `arguments` (an object).
- The result is a **`function_result` step** whose id field is **`call_id`**; `result` is a list of content items.
- Sending the whole history back is the stateless mode, so `store` is `False` (lesson 09's server-side state is the alternative).

In [10]:
def to_interactions_tools(messages: list[dict]) -> dict:
    check_tool_pairs(messages)
    steps = []
    for m in messages:
        if m["role"] == "user":
            steps.append({"type": "user_input", "content": [{"type": "text", "text": m["text"]}]})
        elif m["role"] == "assistant":
            steps += [{"type": "function_call", "id": c["id"], "name": c["name"], "arguments": c["args"]}
                      for c in m["tool_calls"]]
        else:
            steps.append({"type": "function_result", "call_id": m["call_id"], "name": m["name"],
                          "result": [{"type": "text", "text": m["content"]}]})
    tool = {"type": "function", **SEARCH_NOTES_SCHEMA}
    return {"model": VENDOR_MODELS["gemini"], "tools": [tool], "input": steps, "store": False}

interactions_req = to_interactions_tools(history)
for step in interactions_req["input"]:
    print(step["type"], json.dumps(step)[:95])

user_input {"type": "user_input", "content": [{"type": "text", "text": "Which of my notes mention Mars?"}]
function_call {"type": "function_call", "id": "call_465692", "name": "search_notes", "arguments": {"query": "
function_result {"type": "function_result", "call_id": "call_465692", "name": "search_notes", "result": [{"type


In real use you would append the model's `function_call` step exactly as received (with any thought step
before it), as lesson 22 did with `generate_content`. We build it by hand here only to see its shape.

### 8. Do the SDKs accept them?
Same check as lesson 10: each SDK turns our dict into its request body, and the local transport stops it.

In [11]:
claude = anthropic.Anthropic(api_key="unused", max_retries=0, http_client=local_http())
gpt = openai.OpenAI(api_key="unused", max_retries=0, http_client=local_http())
gem = genai.Client(api_key="unused", http_options={"httpx_client": local_http()})
for name, send, req in [("anthropic", claude.messages.create, anthropic_req),
                        ("openai", gpt.responses.create, responses_req),
                        ("gemini", gem.interactions.create, interactions_req)]:
    sent = wire(lambda: send(**req))
    print(f"  {name}: body == our dict: {sent == req}")

POST https://api.anthropic.com/v1/messages
  anthropic: body == our dict: True
POST https://api.openai.com/v1/responses
  openai: body == our dict: True
POST https://generativelanguage.googleapis.com/v1beta/interactions
  gemini: body == our dict: True


### 9. Side by side

| | Anthropic Messages | OpenAI Responses | Gemini Interactions | Gemini `generate_content` (lesson 22) |
|---|---|---|---|---|
| tool schema args | `input_schema` | `parameters` (+ `"type": "function"`) | `parameters` (+ `"type": "function"`) | `FunctionDeclaration` |
| the call | `tool_use` block in `assistant` msg | `function_call` item | `function_call` step | `function_call` part, role `model` |
| call id field | `id` | `call_id` | `id` | `id` |
| arguments | object `input` | JSON **string** `arguments` | object `arguments` | object `args` |
| the result | `tool_result` block in `user` msg | `function_call_output` item | `function_result` step | `function_response` part, role `user` |
| result's id field | `tool_use_id` | `call_id` | `call_id` | `id` |

### 10. What happens when pairing breaks: Gemini `generate_content`
Anthropic documents a 400 for this: *"tool_use ids were found without tool_result blocks immediately
after"* (Anthropic tool-use docs, "Handle tool calls", platform.claude.com/docs/en/agents-and-tools/tool-use).
We have no Anthropic key, so we quote the documented text instead of triggering it. Let's see what Gemini
on Vertex does. First the correct call 2, using lesson 22's recipe.

The two cells below are recorded output: they replay from the cassette and are skipped without one.

In [12]:
model_turn = resp1.candidates[0].content            # appended unchanged (thought signature)
def result_turn(call_id: str) -> types.Content:
    return types.Content(role="user", parts=[types.Part(function_response=types.FunctionResponse(
        id=call_id, name="search_notes", response={"result": hits}))])
good = client.models.generate_content(model=MODEL, contents=[contents[0], model_turn, result_turn(fc.id)],
                                      config=TOOL_CFG)
print("correct pairing ->", good.text)

correct pairing -> Your notes that mention Mars are:

1. **Note 1:** "Mars has two small moons, Phobos and Deimos."
2. **Note 3:** "A day on Mars lasts about 24 hours and 39 minutes."
3. **Note 5:** "The rover Perseverance landed on Mars in February 2021."


Now three broken histories: the result left out (the user just asks the next question), the result
sent with an id the model never issued, and the result placed before the call.

In [13]:
broken = {
    "missing result": [contents[0], model_turn,
                       types.Content(role="user", parts=[types.Part(text="And Venus?")])],
    "wrong id": [contents[0], model_turn, result_turn("call_0")],
    "result first": [contents[0], result_turn(fc.id), model_turn],
}
for label, hist in broken.items():
    try:
        r = client.models.generate_content(model=MODEL, contents=hist, config=TOOL_CFG)
        calls = [(c.name, dict(c.args)) for c in r.function_calls or []]
        print(f"{label:15}-> no error | tool calls: {calls} | text: {(r.text or '')[:60]!r}")
    except Exception as err:
        print(f"{label:15}-> {type(err).__name__}: {str(err)[:120]}")

missing result -> no error | tool calls: [('search_notes', {'query': 'Venus'})] | text: ''
wrong id       -> no error | tool calls: [] | text: 'You have 3 notes that mention Mars:\n\n1. "Mars has two small '
result first   -> ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Requests ending with a model turn are not supported.', 'status


In this recording Vertex accepted the first two broken requests with no error. With the result missing,
the model silently dropped the Mars question and called the tool for Venus. With the wrong id, it
answered anyway: Vertex did not check that `call_0` was never issued. Only the third one failed: with the
result before the call, the history ends with the model turn, and Vertex returned a real 400.

So a broken pairing fails in one of two ways:
- **Loud:** a 400, so you find the bug at once. Anthropic documents one for a missing result; Vertex gave
  one above for the result-first order. (OpenAI's guide shows each `function_call_output` carrying its
  call's `call_id`; we did not test what happens without it.)
- **Silent:** Gemini `generate_content` accepts a missing or unknown-id result, and the bug shows up later
  as a wrong answer or a repeated tool call.

Either way, `check_tool_pairs` catches it before any request leaves Scout.

## At work
- **Adapters must get the details right.** Gateways such as LiteLLM exist to hide these differences
  (track 6); the `id` / `call_id` / `tool_use_id` names and OpenAI's string `arguments` are the details
  every adapter must get right. → lesson 37
- **A crashed tool must still produce a result.** If a tool raises and the loop skips the result, the next
  request is broken; teams always send an error result instead (`is_error: true` on Anthropic). → lesson 28
- **With several calls per turn, every call needs its result**, and on Anthropic all `tool_result`
  blocks go in one `user` message, before any text. → lesson 25
- **Tool results are untrusted text.** Keeping them inside `tool_result` / `function_call_output` blocks,
  never pasted into the system prompt, is the first defence against injection. → lesson 51

## What just happened
- A real Gemini tool call became one neutral `user` / `tool_calls` / `tool` history.
- `check_tool_pairs` enforced the shared rule: each call answered once, by id, right after the call.
- Three converters wrote it as Anthropic `tool_use`/`tool_result` (`tool_use_id`), OpenAI
  `function_call`/`function_call_output` (`call_id`, arguments as a string) and Gemini Interactions
  `function_call`/`function_result` (`call_id`); each SDK sent our dict unchanged.
- On Vertex, a missing or mismatched result gave no error, only a confused model; a result placed before
  the call gave a real 400. Anthropic documents a 400 for a missing result.

## Common mistakes
**1. Putting text before the `tool_result` on Anthropic.** Anthropic's docs say the result blocks must come
first in the `user` message; text first is a documented 400. Our own Anthropic checker enforces both
placement rules. The `ValueError` below is our local imitation, not a vendor response.

In [14]:
def check_anthropic(payload: dict) -> None:
    msgs = payload["messages"]
    for i, m in enumerate(msgs):
        ids = [b["id"] for b in m["content"] if b["type"] == "tool_use"]
        if not ids:
            continue
        nxt = msgs[i + 1]["content"] if i + 1 < len(msgs) else []
        kinds = [b["type"] for b in nxt]
        if "text" in kinds and "tool_result" in kinds[kinds.index("text"):]:
            raise ValueError(f"message {i + 1}: text before tool_result (must come after)")
        got = [b["tool_use_id"] for b in nxt if b["type"] == "tool_result"]
        if sorted(got) != sorted(ids):
            raise ValueError(f"tool_use ids {ids} were found without tool_result blocks immediately after")

Now the mistake: a "Here are the results:" text block put in front of the result.

In [15]:
bad = json.loads(json.dumps(anthropic_req))
bad["messages"][2]["content"].insert(0, {"type": "text", "text": "Here are the results:"})
check_anthropic(bad)

ValueError: message 2: text before tool_result (must come after)

Fix: results first, then any text.

In [16]:
bad["messages"][2]["content"].append(bad["messages"][2]["content"].pop(0))
check_anthropic(bad)
print([b["type"] for b in bad["messages"][2]["content"]])

['tool_result', 'text']


**2. Sending a new user turn before the result.** It is tempting to let the user keep chatting while a
tool runs. The call is then left unanswered, which Anthropic documents as a 400 and which is the silent Gemini bug from step 10. The `ValueError`
below comes from our local checker, not from a vendor.

In [17]:
early = history[:2] + [{"role": "user", "text": "And Venus?"}] + history[2:]
check_tool_pairs(early)

ValueError: message 1: calls ['call_465692'] need results right after, got []

Fix: answer every call first, then add the new user turn.

In [18]:
fixed = history + [{"role": "user", "text": "And Venus?"}]
check_tool_pairs(fixed)
print([m["role"] for m in fixed])

['user', 'assistant', 'tool', 'user']


**3. Passing an object as OpenAI `arguments`.** Anthropic and Gemini take an object; OpenAI wants a JSON
string. A converter that copies the dict through gives the wrong type, and the result's `json.loads` fails
when you read it back.

In [19]:
wrong = {"type": "function_call", "call_id": real_call["id"], "name": "search_notes", "arguments": real_call["args"]}
json.loads(wrong["arguments"])

TypeError: the JSON object must be str, bytes or bytearray, not dict

Fix: `json.dumps` when writing (as `to_responses_tools` does) and `json.loads` when reading a call.

In [20]:
right = responses_req["input"][1]
print(type(right["arguments"]).__name__, json.loads(right["arguments"]))

str {'query': 'Mars'}


## Try it
Lesson 25 will need several calls in one turn. Parallel calls are just several entries in `tool_calls`;
lesson 25 covers when models produce them. Extend the neutral history to two calls in one assistant
turn (`search_notes("Mars")` with id `call_a` and `search_notes("Venus")` with id `call_b`) and their two
results. Convert it with all three converters and print how many `tool_result` blocks are in Anthropic's
last `user` message, and how many `function_call_output` items OpenAI got. Then remove the `call_b` result
and show that `check_tool_pairs` rejects it.

In [21]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [22]:
llm_replay.stop()

## Key terms
- **tool_use block** — an Anthropic content block in an `assistant` message asking for a tool: `id`, `name`, `input`.
- **tool_result block** — an Anthropic content block in a `user` message carrying a tool's output, linked by `tool_use_id`; must come first in that message, right after the call.
- **function_call item** — an OpenAI Responses input/output item for a tool call: `call_id`, `name`, `arguments` (a JSON string).
- **function_call_output** — the OpenAI Responses item that returns a tool's output, linked by `call_id`.
- **function_result step** — the Gemini Interactions step that returns a tool's output, linked by `call_id` to a `function_call` step's `id`.
- **call id matching** — the rule that every tool call is answered by exactly one result carrying its id, placed right after the call.